### Setup

In [83]:
import pandas as pd
import os
import re
from mistralai.client import Mistral
from rapidfuzz import fuzz, process
from tqdm import tqdm
from dotenv import load_dotenv

In [88]:
load_dotenv(dotenv_path='../.env')

True

In [90]:
client = Mistral(api_key=os.getenv('MISTRAL_API_KEY'))

In [31]:
TEMPLATE_OLD_PATH = "../data/submission_template.csv"
TEMPLATE_PATH = "../data/submission_template_v3.csv"

In [32]:
template_df = pd.read_csv(TEMPLATE_PATH)
template_old_df = pd.read_csv(TEMPLATE_OLD_PATH)
template_df['doc_id'] = template_old_df['doc_id']
template_df

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,0,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,0,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,0,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,0,constituency_10_1
4,constituency_10_1_5,พลวัต,0,constituency_10_1
...,...,...,...,...
10048,party_list_34_11_53,ไทยพิทักษ์ธรรม,0,party_list_34_11
10049,party_list_34_11_54,ความหวังใหม่,0,party_list_34_11
10050,party_list_34_11_55,ไทยรวมไทย,0,party_list_34_11
10051,party_list_34_11_56,เพื่อบ้านเมือง,0,party_list_34_11


### EDA

In [33]:
template_df['party_name'].unique()

array(['ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต',
       'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย',
       'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่',
       'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ',
       'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง',
       'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ',
       'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม',
       'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน',
       'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ', nan,
       'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย',
       'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม',
       'สร้างชาติ', 'ใหม่', 'พลังสังคมใหม่', 'สร้างอนาคตไทย',
       'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่',
       'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่',
       'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธ

In [34]:
template_df.head()

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,0,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,0,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,0,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,0,constituency_10_1
4,constituency_10_1_5,พลวัต,0,constituency_10_1


In [35]:
NaN_df = template_df[
    template_df['party_name'].isna()
]

NaN_df

,id,party_name,votes,doc_id
737,constituency_14_2_10,NaN,0,constituency_14_2
738,constituency_14_2_11,NaN,0,constituency_14_2
739,constituency_14_2_12,NaN,0,constituency_14_2
740,constituency_14_2_13,NaN,0,constituency_14_2
741,constituency_14_2_14,NaN,0,constituency_14_2
742,constituency_14_2_15,NaN,0,constituency_14_2
743,constituency_14_2_16,NaN,0,constituency_14_2
744,constituency_14_2_17,NaN,0,constituency_14_2


In [36]:
# 737 - 744 Correctly Null
template_df.iloc[737]

id            constituency_14_2_10
party_name                     NaN
votes                            0
doc_id           constituency_14_2
Name: 737, dtype: object

### Extraction

In [37]:
def thai_num_to_int(text: str) -> int:

    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_to_arabic)

    # remove non-numeric prefixes
    text = re.sub(r"[^\d,]", " ", text)

    match = re.search(r"[\d,]+", text)
    if not match:
        raise ValueError("No numeric value found")

    return int(match.group().replace(",", ""))

In [53]:
KNOWN_PARTIES = [
    'ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต',
    'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย',
    'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่',
    'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ',
    'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง',
    'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ',
    'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม',
    'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน',
    'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ',
    'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย',
    'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม',
    'สร้างชาติ', 'พลังสังคมใหม่', 'สร้างอนาคตไทย',
    'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่',
    'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่',
    'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธรรมใหม่', 'กรีน',
    'แผ่นดินธรรม', 'ประชาไทย', 'ประชาอาสาชาติ',
    'เครือข่ายชาวนาแห่งประเทศไทย', 'ไทยรวมไทย', 'พลังไทยรักชาติ'
]

def is_party(text, threshold=70):
    match = process.extractOne(text, KNOWN_PARTIES, scorer=fuzz.ratio)
    return match and match[1] >= threshold

In [40]:
def extract_party_score_dict(md: str) -> dict:
    result = {}
    party_idx = None
    vote_idx = None

    for line in md.split('\n'):
        if '---' in line or 'รวมคะแนน' in line:
            continue

        parts = [p.strip() for p in line.split('|') if p.strip()]
        if len(parts) < 2:
            continue

        # Auto-detect column positions from first data row with a known party
        if party_idx is None:
            for i, p in enumerate(parts):
                if is_party(p):
                    party_idx = i
                    # Vote column is always the next one after party
                    vote_idx = i + 1
                    break
            if party_idx is None:
                continue

        if len(parts) <= max(party_idx, vote_idx):
            continue

        party = parts[party_idx].strip()
        vote_raw = parts[vote_idx].strip()

        if not is_party(party):
            continue

        try:
            vote = thai_num_to_int(vote_raw)
            if party:
                result[party] = vote
        except Exception:
            pass

    return result

In [ ]:
def extraction(path):
    try:
        if not os.path.exists(path):
            return {}
        filename = os.path.basename(path)
        with open(path, "rb") as f:
            uploaded = client.files.upload(
                file={"file_name": filename, "content": f},
                purpose="ocr"
            )
        signed_url = client.files.get_signed_url(file_id=uploaded.id)
        result = client.ocr.process(
            model="mistral-ocr-latest",
            document={"type": "document_url", "document_url": signed_url.url}
        )
        # Combine all pages markdown
        print(result.pages[0].markdown)
        md = "".join(page.markdown for page in result.pages)
        return extract_party_score_dict(md)
    except Exception as e:
        print(e.__str__())
        return {}

### Inference

In [54]:
def assign_votes(df, vote_dict, threshold=60):
    vote_dict = {k: v for k, v in vote_dict.items() if k != 'รวมคะแนนทั้งสิ้น'}
    keys = list(vote_dict.keys())

    def get_vote(party_name):
        if pd.isna(party_name):
            return 0
        match = process.extractOne(
            party_name,
            keys,
            scorer=fuzz.token_set_ratio  # changed from fuzz.ratio
        )
        if match is None:
            return 0
        best_key, score, _ = match
        return vote_dict[best_key] if score >= threshold else 0

    df['votes'] = df['party_name'].apply(get_vote)
    return df

In [43]:
submission_df = template_df.copy()

In [ ]:
PREFIX = "../pdf/"

doc_ids = submission_df['doc_id'].unique()

In [75]:
for doc_id in tqdm(doc_ids, desc="Processing", unit="doc"):
	try:
		vote_dict = extraction(PREFIX + doc_id + '.pdf')
		print(doc_id, vote_dict)
		mask = submission_df["doc_id"] == doc_id
		submission_df.loc[mask] = assign_votes(
			submission_df.loc[mask].copy(),
			vote_dict,
			threshold=60
		)
	except Exception as e:
		print(f"{doc_id}: {str(e)}")

Processing:   0%|          | 1/300 [00:07<37:13,  7.47s/doc]

constituency_10_1 {'ประชาชน': 34167, 'ประชาธิปไตย': 14813, 'ภูมิใจไทย': 14368, 'เพื่อไทย': 6030, 'รวมไทยสร้างชาติ': 2075, 'โอกาสใหม่': 1133, 'ไทยภักดี': 1023, 'เศรษฐกิจ': 979, 'ไทยสร้างไทย': 629, 'ไทยก้าวใหม่': 489, 'พลวัต': 351, 'กล้าธรรม': 244, 'ปวงชนไทย': 168, 'รักชาติ': 165, 'ทางเลือกใหม่': 154, 'วิชชั่นใหม่': 113, 'ประชาธิปไตยใหม่': 94, 'พลังประชารัฐ': 80}


Processing:   1%|          | 2/300 [00:14<34:58,  7.04s/doc]

constituency_10_10 {'ประชาชน': 41804, 'เพื่อไทย': 19047, 'โอกาสใหม่': 9440, 'ภูมิใจไทย': 7925, 'ประชาธิปัตย์': 6372, 'รวมไทยสร้างชาติ': 2012, 'เศรษฐกิจ': 1437, 'ไทยสร้างไทย': 636, 'ไทยก้าวใหม่': 583, 'กล้าธรรม': 545, 'ประชาธิปไตยใหม่': 503, 'รักชาติ': 495, 'พลวัต': 461, 'ประชากรไทย': 282, 'ความหวังใหม่': 203, 'วิชชั่นใหม่': 168}


Processing:   1%|          | 3/300 [00:21<35:22,  7.15s/doc]

...

party_list_34_1 {'ไทยทรัพย์ทวี': 299, 'เพื่อขาตีไทย': 3027, 'มิติใหม่': 290, 'รวมใจไทย': 567, 'รวมไทยสร้างชาติ': 1860, 'พลวัต': 447, 'ประชาธิปไตยใหม่': 687, 'เพื่อไทย': 20879, 'ทางเลือกใหม่': 423, 'เศรษฐกิจ': 3346, 'เสร็จรถไทย': 520, 'รวมพลังประชาชน': 400, 'ท้องที่ไทย': 23, 'อนาคตไทย': 57, 'พลังเพื่อไทย': 149, 'พลังสังคมใหม่': 22, 'สังคมประชาธิปไตยไทย': 50, 'โทรวมพลัง': 2230, 'ก้าวอิสระ': 37, 'ปวงขนไทย': 24, 'วิชชั้นใหม่': 91, 'เพื่อชีวิตใหม่': 24, 'คะองไทย': 47, 'ประชาธิปไตย': 5058, 'ไทยก้าวหน้า': 126, 'แรงงานสร้างชาติ': 87, 'ประชากรไทย': 129, 'ครูไทยเพื่อประชาชน': 30, 'ประชาชาติ': 104, 'สร้างอนาคตไทย': 60, 'รักชาติ': 80, 'ไทยพร้อม': 77, 'ภูมิใจไทย': 12515, 'พลังธรรมใหม่': 160, 'กรีน': 37, 'ไทยธรรม': 26, 'แผ่นดินธรรม': 26, 'กล้าถรวม': 188, 'พลังประชาธิปไตย': 146, 'โอกาสใหม่': 50, 'เป็นธรรม': 64, 'ประชาชน': 33820, 'ประชาไทย': 113, 'ไทยสร้างไทย': 2056, 'ไทยก้าวใหม่': 340, 'พร้อม': 48, 'เครือข่ายชาวบ้านแห่งประเทศไทย': 25, 'ไทยพิทักษ์ธรรม': 5, 'ความหรังใหม่': 19, 'ไทยรวมไทย': 10, 'เพื่อบ้

Processing: 100%|█████████▉| 299/300 [37:41<00:10, 10.41s/doc]

party_list_34_10 {'เพื่อชาติไทย': 4187, 'รวมใจไทย': 68, 'รวมใจลดร้างชาติ': 759, 'ประชาธิปไตยไหม้': 669, 'เพื่อไทย': 7462, 'เศรษฐกิจ': 6896, 'เสร็จรถไทย': 61, 'รวมพลังประชาชน': 728, 'ท้องที่ไทย': 63, 'อนาคตไทย': 58, 'พลังเพื่อไทย': 172, 'สังคมประชาธิปไตยไทย': 17, 'ฟิวชัน': 175, 'ไทรวมพลัง': 46161, 'ก้าวอิสระ': 321, 'ปวงชนไทย': 74, 'เพื่อชีวิตไหม้': 39, 'คอองไทย': 65, 'ประชาธิปไตย': 1146, 'ไทยก้าวหน้า': 106, 'ไทยภักดี': 172, 'แรงงานสร้างชาติ': 133, 'ประชากรไทย': 477, 'ครูไทยเพื่อประชาชน': 148, 'ประชาชาติ': 85, 'สร้างอนาคตไทย': 58, 'รักชาติ': 72, 'ไทยพร้อม': 75, 'ภูมิใจไทย': 5520, 'พลังธรรมไหม้': 664, 'กรีน': 40, 'ไทยธรรม': 64, 'แผ่นดินธรรม': 51, 'กล้าธรรม': 54, 'พลังประชารัฐ': 35, 'โอกาสไหม้': 18, 'เป็นธรรม': 41, 'ประชาชน': 11557, 'ประชาไทย': 56, 'ไทยสร้างไทย': 666, 'ไทยก้าวไหม้': 89, 'ประชาอาสาชาติ': 557, 'พร้อม': 61, 'เครือข่ายชาวนาแห่งประเทศไทย': 14, 'ไทยพิพิธภัณฑ์ธรรม': 15, 'ความหวังไหม้': 14, 'ไทยรวมไทย': 44, 'เพื่อบ้านไหม้': 34, 'พลังไทยรักชาติ': 81}


Processing: 100%|██████████| 300/300 [37:51<00:00,  7.57s/doc]

party_list_34_11 {'ไทยพร้อมลำไส้': 162, 'เพื่อชาติต่อ': 1552, 'มิติใหม่': 505, 'รวมใจไทย': 361, 'รวมไทยสร้างชาติ': 1670, 'พลวัต': 234, 'ประชาธิปไตยใหม่': 558, 'เพื่อไทย': 13111, 'ทางเลือกใหม่': 134, 'เศรษฐกิจ': 1913, 'เสร็จรวมไทย': 569, 'รวมพลังประชาชน': 506, 'ท้องที่ไทย': 72, 'อนาคตไทย': 85, 'พลังเพื่อไทย': 232, 'ไทยชนะ': 185, 'พลังสังคมใหม่': 30, 'สังคมประชาธิปไตยไทย': 54, 'ฟิวชัน': 63, 'ไตรวมพลัง': 6554, 'ก้าวอิสระ': 71, 'ปวงชนไทย': 115, 'วิชชั่นใหม่': 317, 'เพื่อชีวิตใหม่': 57, 'คอยงไทย': 120, 'ประชาธิปัตย์': 2921, 'ไทยก้าวหน้า': 171, 'แรงงานสร้างชาติ': 134, 'ประชากรไทย': 284, 'ครูไทยเพื่อประชาชน': 255, 'ประชาชาติ': 202, 'สร้างอนาคตไทย': 151, 'รักชาติ': 142, 'ไทยพร้อม': 204, 'ภูมิใจไทย': 16830, 'พลังธรรมใหม่': 300, 'กรีน': 58, 'ไทยธรรม': 110, 'แผ่นดินธรรม': 34, 'กล้าธรรม': 157, 'พลังประชารัฐ': 178, 'โอกาสใหม่': 34, 'เป็นธรรม': 70, 'ประชาชน': 17866, 'ประชาไทย': 162, 'ไทยสร้างไทย': 1146, 'ไทยก้าวใหม่': 143, 'ประชาอาสาชาติ': 64, 'พร้อม': 43, 'เครือข่ายชาวนาแห่งประเทศไทย': 20, 'ไทยพิทั

In [76]:
submission_df

,id,party_name,votes,doc_id
0,constituency_10_1_1,ประชาธิปัตย์,14813,constituency_10_1
1,constituency_10_1_2,ภูมิใจไทย,14368,constituency_10_1
2,constituency_10_1_3,เศรษฐกิจ,979,constituency_10_1
3,constituency_10_1_4,กล้าธรรม,244,constituency_10_1
4,constituency_10_1_5,พลวัต,351,constituency_10_1
...,...,...,...,...
10048,party_list_34_11_53,ไทยพิทักษ์ธรรม,14,party_list_34_11
10049,party_list_34_11_54,ความหวังใหม่,41,party_list_34_11
10050,party_list_34_11_55,ไทยรวมไทย,29,party_list_34_11
10051,party_list_34_11_56,เพื่อบ้านเมือง,41,party_list_34_11


In [77]:
output_df = submission_df.drop(['doc_id', 'party_name'], axis=1)

In [78]:
output_df

,id,votes
0,constituency_10_1_1,14813
1,constituency_10_1_2,14368
2,constituency_10_1_3,979
3,constituency_10_1_4,244
4,constituency_10_1_5,351
...,...,...
10048,party_list_34_11_53,14
10049,party_list_34_11_54,41
10050,party_list_34_11_55,29
10051,party_list_34_11_56,41


In [79]:
output_df.to_csv("submission2.csv", index=False)

In [82]:
output_df[output_df['votes'] == 0].count()

id       1581
votes    1581
dtype: int64